# Stage 2: Brown Shipley Transformation

**Purpose**: Transform Stage 1 raw data into standardized `individual_dfm_consolidated` format

**Replicates Excel Workbook's "Edited" Sheet**:
- Policy mapping (Account Number → Policyholder_Number)
- Security identifier resolution (SEDOL priority, ISIN fallback, synthetic cash codes)
- Asset name enrichment from security master
- Include/Remove flagging with exclusion reasons
- Holdings check flags for validation
- Decision traceability

**Input**: `stage1_brown_shipley_positions_raw`, `stage1_brown_shipley_cash_raw`

**Output**: `individual_dfm_consolidated` (row-for-row with Stage 1, enriched)

**Row Count Expectation**: Same as Stage 1 raw (e.g., 220 in → 220 out, with flags)

In [ ]:
# ========== Parameters ==========
period = "2026-03"            # YYYY-MM
run_id = "manual_test_run"    # passed by orchestrator in real run

try:
    period = mssparkutils.notebook.params.get("period") or period
    run_id = mssparkutils.notebook.params.get("run_id") or run_id
except Exception:
    pass

dfm_id = "brown_shipley"
dfm_name = "Brown Shipley"
profile_id = "brown_shipley_default"

print(f"[STAGE 2] Starting Brown Shipley transformation")
print(f"  Period: {period}")
print(f"  Run ID: {run_id}")
print(f"  DFM: {dfm_name}")

In [ ]:
# ========== Imports ==========
from pyspark.sql import functions as F
from pyspark.sql import Window
from datetime import datetime
import json

print("[STAGE 2] Imports complete")

## Step 1: Read Stage 1 Raw Tables

Read positions and cash data from Stage 1 raw tables (created by v2 ingestion).

In [ ]:
# ========== Read Stage 1 Raw Data ==========
print("\n[STEP 1] Reading Stage 1 raw tables...")
print("=" * 70)

def table_exists(table_name):
    try:
        return spark.catalog.tableExists(table_name)
    except Exception:
        return False

# Read positions (securities)
positions_table = "stage1_brown_shipley_positions_raw"
if not table_exists(positions_table):
    print(f"[ERROR] Table {positions_table} not found. Run nb_ingest_brown_shipley_v2 first.")
    mssparkutils.notebook.exit("FAILED")

positions = (
    spark.table(positions_table)
    .filter((F.col("period") == period) & (F.col("run_id") == run_id))
)
positions_count = positions.count()
print(f"  Positions (securities): {positions_count} rows")

# Read cash
cash_table = "stage1_brown_shipley_cash_raw"
if table_exists(cash_table):
    cash = (
        spark.table(cash_table)
        .filter((F.col("period") == period) & (F.col("run_id") == run_id))
    )
    cash_count = cash.count()
    print(f"  Cash: {cash_count} rows")
else:
    print("  Cash: table not found (will use positions only)")
    cash = None

if positions_count == 0:
    print("[ERROR] No data found for this period/run_id. Check Stage 1 ingestion.")
    mssparkutils.notebook.exit("NO_DATA")

print("\n[STEP 1] Stage 1 data loaded successfully")

## Step 2: Combine Positions + Cash

Full outer join on `client_id` to preserve all rows from both sources.

In [ ]:
# ========== Combine Positions and Cash ==========
print("\n[STEP 2] Combining positions and cash...")
print("=" * 70)

if cash is not None and cash.count() > 0:
    # Full outer join on client_id to preserve all rows
    combined = positions.alias("pos").join(
        cash.alias("cash"),
        F.col("pos.client_id") == F.col("cash.client_id"),
        "full_outer"
    )
    
    # Coalesce common fields
    combined = combined.select(
        # Provenance
        F.coalesce(F.col("pos.period"), F.col("cash.period")).alias("period"),
        F.coalesce(F.col("pos.run_id"), F.col("cash.run_id")).alias("run_id"),
        F.coalesce(F.col("pos.dfm_id"), F.col("cash.dfm_id")).alias("dfm_id"),
        F.coalesce(F.col("pos.source_file"), F.col("cash.source_file")).alias("source_file"),
        F.coalesce(F.col("pos.source_row_id"), F.col("cash.source_row_id")).alias("source_row_id"),
        
        # Policy identifiers
        F.coalesce(F.col("pos.client_id"), F.col("cash.client_id")).alias("client_id"),
        
        # Dates
        F.coalesce(F.col("pos.value_date"), F.col("cash.value_date")).alias("value_date"),
        
        # Security identifiers (from positions only)
        F.col("pos.isin_code").alias("isin"),
        F.col("pos.sedol_code").alias("sedol"),
        F.col("pos.description_of_security").alias("instrument_name"),
        F.col("pos.type_of_financial_instrument").alias("instrument_type"),
        
        # Position values (from positions)
        F.col("pos.balance").alias("balance_local"),
        F.col("pos.accrued_interest").alias("accrued_interest_local"),
        F.col("pos.currency_code").alias("position_currency"),
        
        # Cash values (from cash)
        F.col("cash.accounting_balance").alias("cash_balance_local"),
        F.col("cash.account_currency").alias("cash_currency"),
        
        # Keep all other position columns for reference
        F.col("pos.*").alias("pos_all"),
        F.col("cash.*").alias("cash_all")
    )
else:
    # No cash data - use positions only
    combined = positions.select(
        "period", "run_id", "dfm_id", "source_file", "source_row_id",
        F.col("client_id").alias("client_id"),
        F.col("value_date").alias("value_date"),
        F.col("isin_code").alias("isin"),
        F.col("sedol_code").alias("sedol"),
        F.col("description_of_security").alias("instrument_name"),
        F.col("type_of_financial_instrument").alias("instrument_type"),
        F.col("balance").alias("balance_local"),
        F.col("accrued_interest").alias("accrued_interest_local"),
        F.col("currency_code").alias("position_currency"),
        F.lit(None).cast("string").alias("cash_balance_local"),
        F.lit(None).cast("string").alias("cash_currency")
    )

combined_count = combined.count()
print(f"  Combined rows: {combined_count}")
print(f"  Expected: {positions_count} (row-for-row with Stage 1)")

if combined_count != positions_count:
    print(f"  [WARNING] Row count mismatch. Expected {positions_count}, got {combined_count}")

print("\n[STEP 2] Positions and cash combined successfully")

## Step 3: Load Reference Data

Load policy mappings, security master, and FX rates.

In [ ]:
# ========== Load Reference Data ==========
print("\n[STEP 3] Loading reference data...")
print("=" * 70)

# Policy Mapping
policy_mapping_path = "Files/config/policy_mapping.csv"
try:
    policy_mapping = (
        spark.read.option("header", True).csv(policy_mapping_path)
        .filter(F.col("dfm_id") == dfm_id)
        .select(
            F.col("dfm_policy_ref").alias("client_id_map"),
            F.col("ih_policy_ref").alias("policyholder_number")
        )
    )
    policy_map_count = policy_mapping.count()
    print(f"  Policy mappings: {policy_map_count} rows for {dfm_name}")
except Exception as e:
    print(f"  [WARNING] Could not load policy_mapping.csv: {e}")
    print("  Will use client_id as policyholder_number (no mapping)")
    policy_mapping = None

# Security Master
security_master_path = "Files/config/security_master.csv"
try:
    security_master = (
        spark.read.option("header", True).csv(security_master_path)
        .select(
            F.upper(F.trim(F.col("isin"))).alias("isin_master"),
            F.upper(F.trim(F.col("sedol"))).alias("sedol_master"),
            F.col("asset_name").alias("asset_name_master"),
            F.col("asset_class"),
            F.col("currency_iso")
        )
    )
    security_master_count = security_master.count()
    print(f"  Security master: {security_master_count} rows")
except Exception as e:
    print(f"  [WARNING] Could not load security_master.csv: {e}")
    security_master = None

# FX Rates
fx_rates_path = "Files/config/fx_rates.csv"
try:
    fx_rates = (
        spark.read.option("header", True).csv(fx_rates_path)
        .select(
            F.upper(F.trim(F.col("currency"))).alias("currency"),
            F.col("rate_to_gbp").cast("double").alias("fx_rate")
        )
    )
    fx_count = fx_rates.count()
    print(f"  FX rates: {fx_count} currencies")
except Exception as e:
    print(f"  [WARNING] Could not load fx_rates.csv: {e}")
    fx_rates = None

print("\n[STEP 3] Reference data loaded")

## Step 4: Policy Mapping

Map `client_id` → `policyholder_number` using policy_mapping reference.

In [ ]:
# ========== Apply Policy Mapping ==========
print("\n[STEP 4] Applying policy mapping...")
print("=" * 70)

if policy_mapping is not None:
    enriched = combined.join(
        policy_mapping,
        combined["client_id"] == policy_mapping["client_id_map"],
        "left"
    )
    
    # Track mapping decision
    enriched = enriched.withColumn(
        "policy_mapping_applied",
        F.when(F.col("policyholder_number").isNotNull(), F.lit(True)).otherwise(F.lit(False))
    )
    
    # If no mapping found, use client_id as fallback
    enriched = enriched.withColumn(
        "policyholder_number",
        F.coalesce(F.col("policyholder_number"), F.col("client_id"))
    )
    
    mapped_count = enriched.filter(F.col("policy_mapping_applied") == True).count()
    print(f"  Rows with policy mapping: {mapped_count} / {combined_count}")
    print(f"  Rows using client_id fallback: {combined_count - mapped_count}")
else:
    # No mapping available - use client_id directly
    enriched = combined.withColumn("policyholder_number", F.col("client_id"))
    enriched = enriched.withColumn("policy_mapping_applied", F.lit(False))
    print(f"  Using client_id as policyholder_number (no mapping table)")

print("\n[STEP 4] Policy mapping complete")

## Step 5: Security Identifier Resolution

Priority: SEDOL → ISIN → Synthetic cash code (`TPY_CASH_{CURRENCY}`).

In [ ]:
# ========== Security Identifier Resolution ==========
print("\n[STEP 5] Resolving security identifiers...")
print("=" * 70)

# Determine local currency (prioritize position currency, fallback to cash currency, default GBP)
enriched = enriched.withColumn(
    "local_currency",
    F.upper(F.coalesce(
        F.nullif(F.trim(F.col("position_currency")), ""),
        F.nullif(F.trim(F.col("cash_currency")), ""),
        F.lit("GBP")
    ))
)

# Identify cash lines (no ISIN and no SEDOL)
enriched = enriched.withColumn(
    "_is_cash_line",
    (F.col("isin").isNull() | (F.trim(F.col("isin")) == "")) &
    (F.col("sedol").isNull() | (F.trim(F.col("sedol")) == ""))
)

# Apply identifier resolution logic
enriched = enriched.withColumn(
    "security_code",
    F.when(
        F.col("sedol").isNotNull() & (F.trim(F.col("sedol")) != ""),
        F.upper(F.trim(F.col("sedol")))
    )
    .when(
        F.col("isin").isNotNull() & (F.trim(F.col("isin")) != ""),
        F.upper(F.trim(F.col("isin")))
    )
    .when(
        F.col("_is_cash_line"),
        F.concat(F.lit("TPY_CASH_"), F.col("local_currency"))
    )
    .otherwise(None)
)

# Set ID type
enriched = enriched.withColumn(
    "id_type",
    F.when(
        F.col("sedol").isNotNull() & (F.trim(F.col("sedol")) != ""),
        F.lit("SEDOL")
    )
    .when(
        F.col("isin").isNotNull() & (F.trim(F.col("isin")) != ""),
        F.lit("ISIN")
    )
    .when(
        F.col("_is_cash_line"),
        F.lit("Undertaking - Specific")
    )
    .otherwise(None)
)

# Track which identifier was chosen
enriched = enriched.withColumn(
    "identifier_chosen",
    F.when(
        F.col("sedol").isNotNull() & (F.trim(F.col("sedol")) != ""),
        F.lit("SEDOL")
    )
    .when(
        F.col("isin").isNotNull() & (F.trim(F.col("isin")) != ""),
        F.lit("ISIN")
    )
    .when(
        F.col("_is_cash_line"),
        F.lit("SYNTHETIC_CASH")
    )
    .otherwise(F.lit("NONE"))
)

# Distribution of identifier types
identifier_dist = enriched.groupBy("identifier_chosen").count().orderBy(F.desc("count"))
print("  Identifier resolution distribution:")
for row in identifier_dist.collect():
    print(f"    {row['identifier_chosen']}: {row['count']} rows")

print("\n[STEP 5] Security identifier resolution complete")

## Step 6: Security Master Enrichment

Enrich asset names from security master, with fallback to source instrument name.

In [ ]:
# ========== Security Master Enrichment ==========
print("\n[STEP 6] Enriching from security master...")
print("=" * 70)

if security_master is not None:
    # Join on ISIN first
    enriched = enriched.join(
        security_master,
        F.upper(F.trim(enriched["isin"])) == security_master["isin_master"],
        "left"
    )
    
    # Fallback: if no ISIN match, try SEDOL
    # (would need another join, skip for now - prioritize ISIN)
    
    enriched = enriched.withColumn(
        "asset_name",
        F.when(
            F.col("_is_cash_line"),
            F.concat(F.lit("CASH - "), F.col("local_currency"))
        )
        .otherwise(
            F.upper(F.coalesce(
                F.col("asset_name_master"),
                F.col("instrument_name")
            ))
        )
    )
    
    enriched_count = enriched.filter(F.col("asset_name_master").isNotNull()).count()
    print(f"  Rows enriched from security master: {enriched_count}")
    print(f"  Rows using source instrument name: {combined_count - enriched_count}")
else:
    # No security master - use source instrument name
    enriched = enriched.withColumn(
        "asset_name",
        F.when(
            F.col("_is_cash_line"),
            F.concat(F.lit("CASH - "), F.col("local_currency"))
        )
        .otherwise(F.upper(F.col("instrument_name")))
    )
    print("  Using source instrument names (no security master)")

print("\n[STEP 6] Asset name enrichment complete")

## Step 7: Convert Values and Apply FX

Parse numeric values, apply FX rates, calculate GBP equivalents.

In [ ]:
# ========== Value Conversion and FX Application ==========
print("\n[STEP 7] Converting values and applying FX rates...")
print("=" * 70)

# Helper: convert European decimal format "1.234,56" to "1234.56"
def parse_euro_decimal(col_name):
    return F.regexp_replace(
        F.regexp_replace(F.trim(F.col(col_name)), r"\.", ""),
        ",", "."
    ).cast("double")

# Parse position values
enriched = enriched.withColumn("bid_value_local", parse_euro_decimal("balance_local"))
enriched = enriched.withColumn("accrued_interest_local", parse_euro_decimal("accrued_interest_local"))

# Parse cash values
enriched = enriched.withColumn("cash_value_local", parse_euro_decimal("cash_balance_local"))

# Apply FX rates
if fx_rates is not None:
    enriched = enriched.join(
        fx_rates,
        enriched["local_currency"] == fx_rates["currency"],
        "left"
    )
    
    # For GBP, rate = 1.0
    enriched = enriched.withColumn(
        "fx_rate",
        F.when(F.col("local_currency") == "GBP", F.lit(1.0))
         .otherwise(F.col("fx_rate"))
    )
else:
    # No FX rates - assume everything is GBP
    enriched = enriched.withColumn("fx_rate", F.lit(1.0))

# Calculate GBP values
enriched = enriched.withColumn(
    "bid_value_gbp",
    F.when(F.col("bid_value_local").isNull(), F.lit(None).cast("double"))
     .when(F.col("fx_rate").isNull(), F.lit(None).cast("double"))
     .otherwise(F.col("bid_value_local") * F.col("fx_rate"))
)

enriched = enriched.withColumn(
    "cash_value_gbp",
    F.when(F.col("cash_value_local").isNull(), F.lit(0.0))
     .when(F.col("fx_rate").isNull(), F.lit(None).cast("double"))
     .otherwise(F.col("cash_value_local") * F.col("fx_rate"))
)

enriched = enriched.withColumn(
    "accrued_interest_gbp",
    F.when(F.col("accrued_interest_local").isNull(), F.lit(0.0))
     .when(F.col("fx_rate").isNull(), F.lit(None).cast("double"))
     .otherwise(F.col("accrued_interest_local") * F.col("fx_rate"))
)

# Holdings and price (not available in Brown Shipley source)
enriched = enriched.withColumn("holding", F.lit(None).cast("double"))
enriched = enriched.withColumn("local_bid_price", F.lit(None).cast("double"))

print("  Value conversion complete")
print("  FX rates applied")
print("  GBP equivalents calculated")

print("\n[STEP 7] Value conversion and FX application complete")

## Step 8: Include/Remove Flagging

Apply exclusion rules:
- Zero value with non-zero holdings
- Zero value and zero holdings
- Currency placeholders with no quantity

In [ ]:
# ========== Include/Remove Flagging ==========
print("\n[STEP 8] Applying Include/Remove logic...")
print("=" * 70)

# TODO: Load exclusion_policies table when available
# For now, use simple rule-based exclusions

enriched = enriched.withColumn(
    "exclusion_reason_code",
    F.when(
        (F.coalesce(F.col("bid_value_gbp"), F.lit(0.0)) == 0.0) &
        (F.coalesce(F.col("cash_value_gbp"), F.lit(0.0)) == 0.0),
        F.lit("ZERO_ALL")
    )
    .when(
        F.col("_is_cash_line") &
        (F.coalesce(F.col("cash_value_gbp"), F.lit(0.0)) == 0.0),
        F.lit("CURRENCY_PLACEHOLDER")
    )
    # Add more exclusion rules here as needed
    .otherwise(None)
)

enriched = enriched.withColumn(
    "include_flag",
    F.when(F.col("exclusion_reason_code").isNotNull(), F.lit("Remove"))
     .otherwise(F.lit("Include"))
)

# Distribution
include_dist = enriched.groupBy("include_flag").count().orderBy("include_flag")
print("  Include/Remove distribution:")
for row in include_dist.collect():
    print(f"    {row['include_flag']}: {row['count']} rows")

exclusion_reasons = enriched.filter(F.col("include_flag") == "Remove") \
    .groupBy("exclusion_reason_code").count().orderBy(F.desc("count"))
if exclusion_reasons.count() > 0:
    print("\n  Exclusion reasons:")
    for row in exclusion_reasons.collect():
        print(f"    {row['exclusion_reason_code']}: {row['count']} rows")

print("\n[STEP 8] Include/Remove flagging complete")

## Step 9: Add Check Columns and Decision Trace

Add validation check flags and decision traceability.

In [ ]:
# ========== Check Columns and Decision Trace ==========
print("\n[STEP 9] Adding check columns and decision trace...")
print("=" * 70)

# Holdings check (MV_001): holding × price vs value, 2% tolerance
# NOTE: Brown Shipley doesn't have holding/price, so this will be not_evaluable
enriched = enriched.withColumn(
    "holdings_check_flag",
    F.when(
        F.col("holding").isNull() | F.col("local_bid_price").isNull(),
        F.lit("not_evaluable")
    )
    .when(
        F.col("bid_value_gbp").isNull() | (F.col("bid_value_gbp") == 0.0),
        F.lit("not_evaluable")
    )
    .when(
        F.abs(
            (F.col("holding") * F.col("local_bid_price") * F.col("fx_rate")) - F.col("bid_value_gbp")
        ) / F.col("bid_value_gbp") > 0.02,
        F.lit("fail")
    )
    .otherwise(F.lit("pass"))
)

# Acquisition value check (stub - not available in source)
enriched = enriched.withColumn("acq_value_check_flag", F.lit("not_evaluable"))

# Decision trace JSON
enriched = enriched.withColumn(
    "decision_trace_json",
    F.to_json(
        F.struct(
            F.col("client_id").alias("policy_original"),
            F.col("policyholder_number").alias("policy_mapped"),
            F.col("policy_mapping_applied"),
            F.col("identifier_chosen"),
            F.col("sedol").alias("source_sedol"),
            F.col("isin").alias("source_isin"),
            F.col("security_code"),
            F.col("exclusion_reason_code"),
            F.col("holdings_check_flag")
        )
    )
)

# Data quality flags
enriched = enriched.withColumn(
    "data_quality_flags",
    F.array(
        F.when(F.col("fx_rate").isNull(), F.lit("FX_NOT_AVAILABLE")),
        F.when(F.col("policy_mapping_applied") == False, F.lit("POLICY_NOT_MAPPED")),
        F.when(F.col("holdings_check_flag") == "not_evaluable", F.lit("MV_NOT_EVALUABLE"))
    ).cast("array<string>")
)

# Remove nulls from array
enriched = enriched.withColumn(
    "data_quality_flags",
    F.expr("filter(data_quality_flags, x -> x is not null)")
)

print("  Check columns added")
print("  Decision trace JSON generated")
print("  Data quality flags populated")

print("\n[STEP 9] Check columns and decision trace complete")

## Step 10: Project to Stage 2 Contract

Project to `individual_dfm_consolidated` schema.

In [ ]:
# ========== Project to Stage 2 Contract ==========
print("\n[STEP 10] Projecting to individual_dfm_consolidated schema...")
print("=" * 70)

# Generate row hash for deduplication
enriched = enriched.withColumn(
    "row_hash",
    F.sha2(
        F.concat_ws(
            "|",
            F.col("period"),
            F.col("dfm_id"),
            F.col("policyholder_number"),
            F.coalesce(F.col("security_code"), F.lit("")),
            F.coalesce(F.col("isin"), F.lit("")),
            F.coalesce(F.col("source_row_id"), F.lit(""))
        ),
        256
    )
)

# Parse value_date to date
enriched = enriched.withColumn(
    "report_date",
    F.coalesce(
        F.to_date(F.col("value_date"), "dd/MM/yyyy"),
        F.to_date(F.col("value_date"), "yyyy-MM-dd"),
        F.to_date(F.col("value_date"))
    )
)

# Select Stage 2 contract columns
stage2_output = enriched.select(
    # Provenance
    F.col("period"),
    F.col("run_id"),
    F.col("dfm_id"),
    F.lit(profile_id).alias("profile_id"),
    F.col("source_file"),
    F.col("source_row_id"),
    F.col("row_hash"),
    
    # Policy identifiers
    F.col("policyholder_number"),
    
    # Security identifiers
    F.col("security_code"),
    F.upper(F.trim(F.col("isin"))).alias("isin"),
    F.upper(F.trim(F.col("sedol"))).alias("sedol"),
    F.lit(None).cast("string").alias("other_security_id"),
    F.col("id_type"),
    F.col("asset_name"),
    
    # Position values
    F.col("holding").cast("decimal(28,10)"),
    F.col("local_bid_price").cast("decimal(28,10)"),
    F.col("local_currency"),
    F.col("fx_rate").cast("decimal(28,10)"),
    F.col("cash_value_gbp").cast("decimal(28,10)"),
    F.col("bid_value_gbp").cast("decimal(28,10)"),
    F.col("accrued_interest_gbp").cast("decimal(28,10)"),
    
    # Include/Remove
    F.col("include_flag"),
    F.col("exclusion_reason_code"),
    
    # Decision trace
    F.col("identifier_chosen"),
    F.col("decision_trace_json"),
    F.col("data_quality_flags"),
    
    # Metadata
    F.col("report_date"),
    F.current_timestamp().alias("transformed_at")
)

# Remove duplicates based on row_hash
stage2_output = stage2_output.dropDuplicates(["row_hash"])

output_count = stage2_output.count()
print(f"  Stage 2 output rows: {output_count}")
print(f"  Input rows (Stage 1): {combined_count}")

if output_count != combined_count:
    print(f"  [WARNING] Row count changed. Expected {combined_count}, got {output_count}")

print("\n[STEP 10] Schema projection complete")

## Step 11: Write to individual_dfm_consolidated

Persist Stage 2 output table.

In [ ]:
# ========== Write Stage 2 Output ==========
print("\n[STEP 11] Writing to individual_dfm_consolidated table...")
print("=" * 70)

try:
    stage2_output.write.mode("append").saveAsTable("individual_dfm_consolidated")
    print(f"  ✓ Successfully wrote {output_count} rows to individual_dfm_consolidated")
except Exception as e:
    print(f"  [ERROR] Failed to write to individual_dfm_consolidated: {e}")
    mssparkutils.notebook.exit("FAILED")

print("\n[STEP 11] Stage 2 output written successfully")

## Summary and Validation

Final statistics and validation checks.

In [ ]:
# ========== Summary ==========
print("\n" + "=" * 70)
print("STAGE 2 TRANSFORMATION COMPLETE")
print("=" * 70)

print(f"\nPeriod: {period}")
print(f"Run ID: {run_id}")
print(f"DFM: {dfm_name}")

print(f"\nRow Counts:")
print(f"  Stage 1 input (positions): {positions_count}")
if cash is not None:
    print(f"  Stage 1 input (cash): {cash_count}")
print(f"  Combined: {combined_count}")
print(f"  Stage 2 output: {output_count}")

include_summary = stage2_output.groupBy("include_flag").count().collect()
print(f"\nInclude/Remove Summary:")
for row in include_summary:
    print(f"  {row['include_flag']}: {row['count']} rows")

print(f"\nNext Steps:")
print(f"  1. Run nb_stage3_tpir_projection to create tpir_load_equivalent")
print(f"  2. Run nb_validate to generate dq_results and dq_exception_rows")
print(f"  3. Review validation results before publish")

print("\n" + "=" * 70)
mssparkutils.notebook.exit("OK")